# Defining Boundary Conditions

Written for dolfinx v0.7.2, python 3.10.6

With the mesh completed, the next step is to define the values of the fields at the boundary of the mesh.

# Libraries / Packages

In [35]:
from mpi4py import MPI # This is the communicator (or message passing interface) that allows python to work in parallel

# Create the MPI communicator (This has to happen before any other code is run.)
comm = MPI.COMM_WORLD
print(f"size = {comm.Get_size()}")
print(f"rank = {comm.Get_rank()}")

import numpy as np # For working with arrays
import dolfinx # Dolfinx is FEniCSx's computation backend / user interface
import leopart.cpp as pyleopart
import leopart.io
import json # For mapping entity names to physical groups
import ufl
import dolfinx.fem.petsc # This has to be done separately as it has optional dependencies
import pyvista # For plotting B field
import sys

size = 1
rank = 0


# Variables & Parameters

In [36]:
from variables import constants, plasma_vars, collimator_vars, plotting_vars, anders_vars

# Input Pre-Defined Mesh

This is collimator_tetra.msh file that was defined in gmsh_collimator.ipynb. GMSH is separate to FEniCSx, and FEniCSx requires the mesh to be in a xmdf format. The file is converted using meshio. 

In [37]:
# Read mesh from the XDMF file (it automatically reads the linked .h5 file)
with dolfinx.io.XDMFFile(comm, "meshes/collimator_tetra.xdmf", "r") as xdmf: 
    mesh = xdmf.read_mesh(name="Grid") 
    print("mesh read") 
    cell_tags = xdmf.read_meshtags(mesh, name="Grid") 
    print("cell tags read")
mesh.topology.create_connectivity(mesh.topology.dim, mesh.topology.dim - 1)
with dolfinx.io.XDMFFile(comm, "meshes/collimator_tetra_facets.xdmf", "r") as xdmf:
    facet_tags = xdmf.read_meshtags(mesh, name="Grid")
    print("facet tags read")

mesh read
cell tags read
facet tags read


# Read in Physical Group Tags

The physical groups are identified by a tag, not by name. To make the code more readable, we map these tags to names. 
To retrieve the tag of the anode, for example, use `physicalgroups['anode']['tag']`

In [38]:
with open("physicalgroups.json", "r") as file:
    physicalgroups = json.load(file)

print("Named physical groups:")
for group, properties in physicalgroups.items():
    print(f"{group}: dim = {properties['dim']}, tag = {properties['tag']}")

Named physical groups:
points: dim = 0, tag = 63
lines: dim = 1, tag = 64
collimator_surface: dim = 2, tag = 54
anode_surface: dim = 2, tag = 55
cathode_surface: dim = 2, tag = 56
trigger_surface: dim = 2, tag = 57
magnets_surface_north: dim = 2, tag = 58
magnets_surface_south: dim = 2, tag = 59
area_surface: dim = 2, tag = 60
trig_no_cathode_surface: dim = 2, tag = 61
outer_surface: dim = 2, tag = 62
collimator: dim = 3, tag = 1
anode: dim = 3, tag = 2
cathode: dim = 3, tag = 3
trig_no_cathode: dim = 3, tag = 4
trigger: dim = 3, tag = 5
magnets_north: dim = 3, tag = 6
magnets_south: dim = 3, tag = 7
area_around_collimator: dim = 3, tag = 8


# Calculate static magnetic field due to magnets

First, the static magnetic field due to the magnets, cathode and anode must be determined. 

To do this, the variational form of Maxwell's equations must be determined. 

To determine the static B field, the two relevant equations are:

$$
\vec{\nabla} \cdot \vec{B} = 0 \tag{1}
$$

$$
\vec{\nabla} \times \vec{H} = \frac{\partial \vec{D}}{\partial t} + \vec{J} \tag{2}
$$

The static $\vec{B}$ field is time-invariant by definition($\therefore \frac{\partial \vec{D}}{\partial t} = 0$), and there are no moving charges ($ \therefore \vec{J} = 0$), so therefore:

$$
\vec{\nabla} \times \vec{H} = 0
$$

If the curl of a vector is zero, then it can be described as the gradient of a scalar potential. In the case of $\vec{H}$, this is the scalar magnetic potential $\phi$. 

$$
\vec{H} = - \vec{\nabla} \phi
$$

We also know that the magnetic field around a permanent magnet can be described by:

$$ 
\vec{B} = \mu_0 (\vec{H} + \vec{M})
$$

Where $\vec{H}$ is the magnetic field strength, and $\vec{M}$ is the magnetisation (or, density of magnetic dipole moments in a magnetic material).

$$
\therefore \vec{H} = \frac{1}{\mu_0} \vec{B} - \vec{M}
$$
$$
\therefore \vec{\nabla} \times \vec{H} = \vec{\nabla} \times (\frac{1}{\mu_0} \vec{B} - \vec{M}) = 0
$$

Since $\vec{\nabla} \cdot \vec{B} = 0$ everywhere, then $\vec{B}$ can be described as the curl of a vector potential field, $\vec{A}$. 

$$
\therefore \vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}) - \vec{M}) = 0
$$

The cross product is distributive over vector addition, so:

$$
\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A})) = \vec{\nabla} \times \vec{M}
$$

To find the weak form, we then multiply both sides by some arbitrary test function $v$ in the function space $V$:

$$
(\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))) \cdot v = (\vec{\nabla} \times \vec{M}) \cdot v
$$

Then integrate over the collimator space $\Omega$:

$$
\int_{\Omega} (\vec{\nabla} \times (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))) \cdot v \ dA = \int_{\Omega} (\vec{\nabla} \times \vec{M}) \cdot v \ dA
$$

We can rearrange the left hand side using this proerty of the divergence and curl:

$$
\vec{v}\cdot(\nabla\times\vec{F}) = \vec{F}\cdot(\nabla\times\vec{v}) + \nabla\cdot(\vec{F}\times\vec{v})
$$

Letting $(\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A}))$ be $\vec{F}$, the left side of the equation is therefore:

$$
\int_\Omega \vec{v}\cdot(\nabla\times\vec{F})\,dx = \int_\Omega \vec{F}\cdot(\nabla\times\vec{v})\,dx + \int_\Omega \nabla\cdot(\vec{F}\times\vec{v})\,dx
$$

The second component here is the divergence of a field integrated over a volume, so can be changed to a surface integral using the divergence theorem:

$$
\int_\Omega (\nabla\times\vec{F})\cdot\vec{v}\,dx = \int_\Omega \vec{F}\cdot(\nabla\times\vec{v})\,dx + \oint_{\partial\Omega}(\vec{F}\times\vec{v})\cdot\vec{n}\,ds
$$

If we impose the condition that $\vec{A} = 0$ on the boundary of the problem, then the left hand side becomes:

$$
\int_\Omega \vec{F}\cdot(\nabla\times\vec{v})\,dx
$$

Doing the same to the right hand side, and setting the magnetisation field to zero outside of the magnets, we then have:

$$
\int_\Omega \vec{M}\cdot(\nabla\times\vec{v})\,dx - \oint_{\partial\Omega}(\vec{M}\times\vec{v})\cdot\vec{n}\,ds
$$

$$
=\int_\Omega \vec{M}\cdot(\nabla\times\vec{v})\,dx
$$

$$
\therefore \int_\Omega (\frac{1}{\mu_0} (\vec{\nabla} \times \vec{A})) \cdot(\nabla\times\vec{v})\,dx = \int_\Omega \vec{M}\cdot(\nabla\times\vec{v})\,dx
$$

This is the weak form we use as input for FEniCSx. 


The values of $\mu_0$ and $\vec{M}$ are then defined on piecewise constant functions:

In [39]:
# Define a function space for the magnetic scalar potential
V = dolfinx.fem.functionspace(mesh, ("N1curl", 1)) # Nedelec space of degree 1 - often used for Maxwell's eqns

# The magnetic permeability of the magnets and vacuum are saved as a piecewise constant function space.
Q = dolfinx.fem.functionspace(mesh, ("DG", 0)) 
mu = dolfinx.fem.Function(Q)

magnet_cells_north = cell_tags.find(physicalgroups['magnets_north']['tag'])
magnet_cells_south = cell_tags.find(physicalgroups['magnets_south']['tag'])
collimator_cells = cell_tags.find(physicalgroups['collimator']['tag'])
cathode_cells = cell_tags.find(physicalgroups['cathode']['tag'])
anode_cells = cell_tags.find(physicalgroups['anode']['tag'])
trigger_cells = cell_tags.find(physicalgroups['trigger']['tag'])
trigger_no_cathode_cells = cell_tags.find(physicalgroups['trig_no_cathode']['tag'])
area_cells = cell_tags.find(physicalgroups['area_around_collimator']['tag'])

mu.x.array[magnet_cells_north] = constants.mu0 * constants.mu_r_magnet # Define magnetic permeability for the magnets and vacuum
mu.x.array[magnet_cells_south] = constants.mu0 * constants.mu_r_magnet
mu.x.array[collimator_cells] = constants.mu0
mu.x.array[cathode_cells] = constants.mu0
mu.x.array[anode_cells] = constants.mu0
mu.x.array[trigger_cells] = constants.mu0
mu.x.array[trigger_no_cathode_cells] = constants.mu0
mu.x.array[area_cells] = constants.mu0

W = dolfinx.fem.functionspace(mesh, ("DG", 0, (3,))) # Discontinuous Galerkin function space with order 0 and ? dimensions 3,?
M = dolfinx.fem.Function(W) # Function M, magnetisation vector

print("mu min/max:", mu.x.array.min(), mu.x.array.max())
print("cells covered:", len(magnet_cells_north) + len(magnet_cells_south) + len(collimator_cells) + len(cathode_cells) + len(anode_cells) + len(trigger_cells) + len(area_cells) + len(trigger_no_cathode_cells),
      "vs total cells:", mesh.topology.index_map(mesh.topology.dim).size_local)

Mmag = constants.Br / (constants.mu0 * constants.mu_r_magnet) # A/m (B_total = mu_0 (H + M), therefore M = B_r / mu_0 in absence of H field)

M_dofs_north = dolfinx.fem.locate_dofs_topological(W, mesh.topology.dim, magnet_cells_north)
M_dofs_south = dolfinx.fem.locate_dofs_topological(W, mesh.topology.dim, magnet_cells_south)

n_mag = collimator_vars.n_mag # number of magnet pairs in the mesh script, so there are 2*n_mag magnets
dtheta = np.pi / n_mag # angular spacing between neighbouring magnets

M_array = M.x.array.reshape(-1, 3)
M_array[:] = 0.0

# Each magnet is magnetised along its own radial direction, with polarity alternating around the ring.
for cells, polarity in ((magnet_cells_north, +1), (magnet_cells_south, -1)):
    centres = dolfinx.mesh.compute_midpoints(mesh, mesh.topology.dim, cells)
    phi = np.arctan2(centres[:, 2], centres[:, 1]) # angle about the x (thruster) axis, from +y towards +z
    phi_mag = np.round(phi / dtheta) * dtheta # snap each cell to the centre line of its own magnet
    radial = np.zeros_like(centres)
    radial[:, 1] = np.cos(phi_mag) # y component of the outward radial direction
    radial[:, 2] = np.sin(phi_mag) # z component (no x component)
    M_array[cells, :] = polarity * Mmag * radial # uniform within each magnet

M.x.array[:] = M_array.flatten()
M.x.scatter_forward()

# So we have three function spaces; V (with A & v), for the magnetic vector potential, Q (with mu) for the magnetic permeability, and W (with M) for the magnetisation

mu min/max: 1.2566370614359173e-06 1.3194689145077132e-06
cells covered: 842996 vs total cells: 842996


The weak form of the magnetic vector equation are then input into FEniCSx and solved to determine the static magnetic field at each point on the model:

In [40]:
A = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a = (1 / mu) * ufl.inner(ufl.curl(A), ufl.curl(v)) * ufl.dx
L = ufl.inner(M, ufl.curl(v)) * ufl.dx   # zero outside magnet automatically

# Zero tangential A on the outer boundary (far-field truncation)
outer_tag = physicalgroups['outer_surface']['tag']
outer_facets = facet_tags.find(outer_tag)
outer_dofs = dolfinx.fem.locate_dofs_topological(V, mesh.topology.dim - 1, outer_facets) # When FENiCSx says "dofs"/"degrees-of-freedom", that should be taken to mean "nodes / mesh cell edges"
A_0 = dolfinx.fem.Function(V) # Functions are automatically initialised to all zeros. Want A = 0 on the boundary
bc = dolfinx.fem.dirichletbc(A_0, outer_dofs)

problem = dolfinx.fem.petsc.LinearProblem(
    a, L, bcs=[bc],
    petsc_options={"ksp_type": "cg", # Krylov space solver = conjugate gradient
                   "ksp_rtol": 1e-8}
    # note: pc_type / pc_hypre_type is deliberately set here
)

ksp = problem.solver
pc = ksp.getPC()

# Force the PC to hypre/ams before giving it any geometry
pc.setType("hypre") # Preconditioner type = use hypre's PC library
pc.setHYPREType("ams") # From hypre's list of preconditioners, use auxillary space Maxwell solver.

# discrete gradient: P1 Lagrange -> degree-1 Nedelec
P1 = dolfinx.fem.functionspace(mesh, ("Lagrange", 1))
G = dolfinx.cpp.fem.petsc.discrete_gradient(P1._cpp_object, V._cpp_object)
G.assemble()
pc.setHYPREDiscreteGradient(G)
pc.setHYPRESetBetaPoissonMatrix(None)

# Vertex coordinates
gdim = mesh.geometry.dim
num_vertices_local = mesh.topology.index_map(0).size_local
coords = mesh.geometry.x[:num_vertices_local, :gdim].copy()
pc.setCoordinates(coords)

# Finish setting the solver options now that the ksp type is set correctly
ksp.setFromOptions()

A_h = problem.solve() # solve for magnetic vector potential A

#----------------------------
# Recover the B field (from https://jsdokken.com/dolfinx-tutorial/chapter3/em.html#visualizing-the-magnetic-field)
W_dg = dolfinx.fem.functionspace(mesh, ("DG", 0, (3,)))
B = dolfinx.fem.Function(W_dg)
B_expr = dolfinx.fem.Expression(ufl.curl(A_h), W_dg.element.interpolation_points())
B.interpolate(B_expr)

# CHECKS:
print("M max:", np.abs(M.x.array).max()) # Magnetisation set correctly (want Br / mu)
print("outer BC dofs found:", len(outer_dofs)) # Boundary conditions applied (want >0)
print("KSP converged reason:", ksp.getConvergedReason()) # Converged reason (want >0)
print("KSP iterations:", ksp.getIterationNumber()) 
print("A_h max:", np.abs(A_h.x.array).max()) # A field solution (want nonzero)
print("B max:", np.abs(B.x.array).max()) # B field (want nonzero)



M max: 909456.8176679733
outer BC dofs found: 0
KSP converged reason: 2
KSP iterations: 23
A_h max: 2.255204542552124e-06
B max: 1.551059265782207


# Plot the background magnetic field

In [41]:
_nb_stdout = sys.stdout          # save the notebook's stream (do this before plotting, because pyvista changes the output, causing print statements to not work otherwise)
# Interpolate our function space V (which is a Nedelec function space) to a Lagrangian space for plotting
V_viz = dolfinx.fem.functionspace(mesh, ("Lagrange", 1, (gdim,)))
B_viz = dolfinx.fem.Function(V_viz)
B_viz.interpolate(B)

# Build the PyVista grid and interpolate values onto it from the mesh
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_viz)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

B_values = B_viz.x.array.real.reshape(-1, gdim)

grid["B"] = B_values
grid["B_mag"] = np.linalg.norm(B_values, axis=1)
grid.set_active_vectors("B")
grid.set_active_scalars("B_mag")

# Build a separate grid of just the magnet cells
mesh_topology, mesh_cell_types, mesh_geometry = dolfinx.plot.vtk_mesh(mesh, mesh.topology.dim)
mesh_grid = pyvista.UnstructuredGrid(mesh_topology, mesh_cell_types, mesh_geometry)

marker = np.zeros(mesh.topology.index_map(mesh.topology.dim).size_local, dtype=np.int32)
marker[cell_tags.find(physicalgroups["magnets_north"]["tag"])] = 1
marker[cell_tags.find(physicalgroups["magnets_south"]["tag"])] = 1

mesh_grid.cell_data["is_magnet"] = marker # Set a mask that has a value of 1 if a given cell is in a magnet, and 0 otherwise
magnet_grid = mesh_grid.threshold(0.5, scalars="is_magnet") # Apply mask to "is_magnet" cells

# Create glyphs (arrows / quiver plot), then downsample them to make the plot more readable
# A glyph is an object / shape places at every cell and scaled and oriented based on some variables (B_mag and B in this case)
glyphs = grid.glyph(orient="B", scale="B_mag", factor=plotting_vars.glyph_size, tolerance=0.000003)

# Plot magnet body and fields (as quiverplot)
plotter = pyvista.Plotter(off_screen=plotting_vars.show_plot)
plotter.add_mesh(magnet_grid, color="dimgray", opacity=0.85, label="Magnet")
sargs = dict(height=0.8, vertical=True, position_x=0.85, position_y=0.1, title=f"|B| (T)\n")
plotter.add_mesh(
    glyphs, scalars="B_mag", cmap="plasma", show_scalar_bar=True,
    scalar_bar_args=sargs
)
plotter.add_title("Magnetic field around collimator magnets\n", font_size=10)
plotter.add_mesh(grid.outline(), color="black", line_width=1)
plotter.add_axes()
plotter.view_isometric()

if plotting_vars.show_plot:
    plotter.screenshot("images/B_field.png")
    print("Saved images/B_field.png")
else:
    plotter.show()

# ---------------------------------------------------------------------
# # Optional: Slice through z-axis, if plot is too noisy
# slice_plane = grid.slice(normal="z", origin=grid.center)
# slice_glyphs = slice_plane.glyph(orient="B", scale="B_mag", factor=plotting_vars.glyph_size)

# plotter2 = pyvista.Plotter(off_screen=plotting_vars.show_plot)
# plotter2.add_mesh(magnet_grid, color="dimgray", opacity=0.4)
# plotter2.add_mesh(
#     slice_glyphs, scalars="B_mag", cmap="plasma", show_scalar_bar=True,
#     scalar_bar_args={"title": "|B| (T)"}

# )
# plotter2.view_xy()
# plotter2.add_axes()

# if plotting_vars.show_plot:
#     plotter2.screenshot("images/B_field_slice.png")
#     print("Saved images/B_field_slice.png")
# else:
#     plotter2.show()
# plotter2.close()
plotter.close()

Saved images/B_field.png


In [42]:
# Put B into a grid form for plotting
tdim = mesh.topology.dim
topo, ctypes, geom = dolfinx.plot.vtk_mesh(mesh, tdim)
cgrid = pyvista.UnstructuredGrid(topo, ctypes, geom)
Bc = B.x.array.real.reshape(-1, 3)
cgrid.cell_data["B"] = Bc
cgrid.cell_data["B_mag"] = np.linalg.norm(Bc, axis=1)

# find the magnets' long axis (I think this is y but pyvista and fenics seem to define the axes differently.)
b = np.array(magnet_grid.bounds).reshape(3, 2)
span_b = b[:, 1] - b[:, 0]
k = 0 #int(np.argmax(span_b)) # Get the index of the long axis
i, j = [a for a in range(3) if a != k] # Other axes' indices. Plot along this plane
c = np.array(magnet_grid.center) # 3d coordinates of centre
print("long axis:", "xyz"[k], "| slice plane spans:", "xyz"[i], "xyz"[j])

# Make a sampling grid. Populate this with B field values from the mesh for plotting in pyvista
pad = 0.5 * max(span_b[i], span_b[j]) 
n = 150 # number of points
u = np.linspace(b[i, 0] - pad, b[i, 1] + pad, n) # Add empty space to either side of the data to make it look nicer
w = np.linspace(b[j, 0] - pad, b[j, 1] + pad, n)
U, Wg = np.meshgrid(u, w) 
normal_vec = [0, 0, 0]; normal_vec[k] = 1

bodies = magnet_grid.connectivity() # labels each separate magnet with RegionId
for r in np.unique(bodies["RegionId"]):
    bb = np.array(bodies.extract_cells(bodies["RegionId"] == r).bounds).reshape(3, 2)
    print(f"magnet {r}: {'xyz'[k]} from {bb[k,0]:.4g} to {bb[k,1]:.4g}") # Print the magnet locations to the screen

def plot_slice(pos: float, fname: str, quantity: str = "B_rad"):
    '''
    Plots the magnetic field on the plane perpendicular to axis k (k = 0, so the y-z plane).
    Args:
        - pos: float, x-position of the slice (thrust axis). c[0] is mid-length.
        - fname: string, the pyvista plot will be saved to this filename
        - quantity: "B_rad" (signed radial component) or "B_mag" (magnitude, log scale)
    '''
    pts = np.zeros((U.size, 3))
    pts[:, i] = U.ravel()
    pts[:, j] = Wg.ravel()
    pts[:, k] = pos

    sampled = pyvista.PolyData(pts).sample(cgrid)
    sampled = sampled.extract_points(sampled["vtkValidPointMask"].astype(bool))

    # Signed radial component about the thrust axis (x). Valid for k = 0 only.
    P = sampled.points
    y, z = P[:, 1] - c[1], P[:, 2] - c[2]
    r = np.hypot(y, z) + 1e-12
    B = sampled["B"]
    sampled["B_rad"] = (y * B[:, 1] + z * B[:, 2]) / r

    origin = c.copy(); origin[k] = pos
    magnet_slice = magnet_grid.slice(normal=normal_vec, origin=origin)

    pl = pyvista.Plotter(off_screen=plotting_vars.show_plot)
    if quantity == "B_rad":
        sargs = dict(height=0.8, vertical=True, position_x=0.1, position_y=0.1, title="B_rad (T)\n")
        pl.add_mesh(sampled, scalars="B_rad", cmap="plasma", #, clim=[-0.1, 0.1]
                    point_size=6, scalar_bar_args=sargs)
        title = "Radial component of B"
    else:
        sargs = dict(height=0.8, vertical=True, position_x=0.1, position_y=0.1, title="|B| (T)\n")
        pl.add_mesh(sampled, scalars="B_mag", cmap="plasma", log_scale=True,
                    clim=[1e-4, 1.72], point_size=6, scalar_bar_args=sargs)
        title = "Magnitude of B"

    sub = sampled.extract_points(np.arange(0, sampled.n_points, 12))
    pl.add_mesh(sub.glyph(orient="B", scale=False, factor=0.05 * pad), color="white")

    if magnet_slice.n_points > 0:
        pl.add_mesh(magnet_slice, color="gray")
    else:
        print(f"No magnets intersect at {'xyz'[k]} = {pos:.4g}")

    {0: pl.view_yz, 1: pl.view_xz, 2: pl.view_xy}[k]()
    pl.add_title(f"{title} within the collimator\n({'xyz'[i]}{'xyz'[j]} plane, {'xyz'[k]} = {pos:.4g})\n",
                 font_size=10)
    pl.screenshot(fname) if plotting_vars.show_plot else pl.show()
    pl.close()

# pick a position inside the magnets' range, e.g.:
plot_slice(0.5 * (b[k, 0] + b[k, 1]), "images/B_slice_mid.png")

# or a few positions at once:
for frac in (0.25, 0.5, 0.75):
    plot_slice(b[k, 0] + frac * span_b[k], f"images/B_slice_{frac:.2f}.png")
    plot_slice(b[k, 0] + frac * span_b[k], f"images/B_slice_mag_{frac:.2f}.png", "B_mag")

sys.stdout = _nb_stdout # Set program to print to jupyter notebook again

long axis: x | slice plane spans: y z
magnet 0: x from 0.0425 to 0.1225
magnet 1: x from 0.0425 to 0.1225
magnet 2: x from 0.0425 to 0.1225
magnet 3: x from 0.0425 to 0.1225
magnet 4: x from 0.0425 to 0.1225
magnet 5: x from 0.0425 to 0.1225


# Define Particles

Once the boundary conditions have been defined, then particles are distributed randomly within the thruster. 

They are assigned an initial position and velocity.

In [43]:
# We want to place these on the trigger surface, so we retrieve the cells that sit on the trigger:

# Get connectivity from facets to cells
facet_to_cell = mesh.topology.connectivity(mesh.topology.dim - 1, mesh.topology.dim)
trigger_no_cathode_cells = list(np.unique(cell_tags.find(physicalgroups['trig_no_cathode']['tag'])))
# TODO would be nice to just have the surface files eventually

n_cells = len(trigger_no_cathode_cells)
n_p_per_cell = list((np.ones(n_cells) * np.ceil(plasma_vars.n_p / n_cells)).astype(int)) # Number of ions per cell
print("Placing particles in cell")
print(f"Number of cells in trigger end = {n_cells}, total number of ions = {plasma_vars.n_p}, \nresulting in {n_p_per_cell[0]} ions per cell")

# xp is the position vector of every particle, and p2cell maps their location to which cell they belong to. 
xp, p2cell = pyleopart.mesh_fill(mesh._cpp_object, n_p_per_cell, trigger_no_cathode_cells)
print(f"num paticles: {xp.shape[0]}")
ptcls = pyleopart.Particles(xp, p2cell)

psip = np.zeros(xp.shape[0]) # Particle property scalar. This must be constant. 

# We also want to assign their velocity, and this can be done using a velocity field
ptcls.add_field("v", [gdim]) # name + value shape, matching Field.value_shape
# vp = ptcls.field("v").data() # shape (n_particles, gdim)
ptcls.field("v").data()[:] = anders_vars.distribute_particles_uncollimated(sum(n_p_per_cell), plasma_vars.v_ion_cathode) # Particle velocity vector

Placing particles in cell
Number of cells in trigger end = 21, total number of ions = 1000, 
resulting in 48 ions per cell
num paticles: 1008


In [44]:
# Plot the inital position of the plasma particles

# mesh -> pyvista grid
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# particles -> point cloud (mesh_fill already returns (N, 3) in 3D)
pts = np.asarray(xp)
cloud = pyvista.PolyData(pts)

# initial velocities as a vector array on the cloud
vel = np.array(ptcls.field("v").data())    # copy
cloud["v"] = vel
cloud["thrust"] = vel[:, 0] # z-component of thrust (order of coordinates is weird)

surfaces = {
    int(tag): grid.extract_cells(cell_tags.find(tag)).extract_surface()
    for tag in np.unique(cell_tags.values)
}

filled = grid.extract_cells(trigger_no_cathode_cells)
outline = grid.extract_surface()

t = 0 # ms, time t of simulation

# labels = {1: "Cathode", 2: "Trigger", 3: "Collimator"}   # your physical group names from gmsh
#render_particles(self, fname, view, cloud, t, surfaces=None, labels=None):
plotting_vars.render_particles("particles_iso.png", "view_isometric", cloud=cloud, t=t,
                               surfaces=surfaces)
# plotting_vars.render_particles("particles_iso.png", "view_isometric", filled, outline, cloud, t)
plotting_vars.render_particles("particles_xy.png", "view_xy", cloud, t, surfaces=surfaces)
plotting_vars.render_particles("particles_xz.png", "view_xz", cloud, t, surfaces=surfaces)
plotting_vars.render_particles("particles_yz.png", "view_yz", cloud, t, surfaces=surfaces)

# Also render the vectors:
arrows = cloud.glyph(orient="v", scale=False, factor=0.05)  # factor = arrow length in mesh units
p = pyvista.Plotter(off_screen=True, window_size=(1400, 1000))
p.add_mesh(outline, style="wireframe", color="lightgrey", opacity=0.3)
sargs = dict(width=0.8, vertical=False, position_x=0.1, position_y=0, title=f"thrust (ms^-1)\n")
for n, (tag, surf) in enumerate(surfaces.items()):
    p.add_mesh(surf, color="lightgrey", opacity=0.1)
p.add_mesh(arrows, scalars="thrust", cmap="plasma", show_scalar_bar=True,
    scalar_bar_args=sargs)
p.view_xy()
p.add_title("Initial positions and velocities of plasma ions")
p.screenshot("images/velocities.png")
p.close()

# Moving the particles

The force $\vec{F}$ due to magnetic and electric fields $\vec{B}$ and $\vec{E}$ acting on a charge q with velocity $\vec{v}$ is given by:

$$
\vec{F} = q(\vec{E} +(\vec{v} \times \vec{B}))
$$

Therefore their velocity at time dt is given by:

$$
m \frac{d\vec{v}}{dt} = q(\vec{E} +(\vec{v} \times \vec{B}))
$$

$$
\therefore \frac{d\vec{v}}{dt} = \frac{q}{m}(\vec{E} +(\vec{v} \times \vec{B}))
$$

Where dt is so small that we assume that $\vec{E}$ and $\vec{B}$ are unchanged over dt, and the movement of the particle over dt is treated as linear. 

The position of the particle after time t is given as:

$$ 
\frac{d\vec{x}}{dt} = \vec{v}
$$

These coupled linear PDEs can then be solved together. 

LeoPart requires three variables that characterise the particles:

$$
\frac{d\vec{x}}{dt} = a_h(\vec{x}, t) = \vec{v}
$$

Where $a_h(\vec{x}, t)$ is a vector-valued velocity field;

$$ 
\frac{d\psi}{dt} = 0
$$
Where $\psi$ is some constant that characterises the particles. In our case, this will be $\frac{q_{ion}}{m_{ion}}$, which is the charge and mass of the molybdenum ions. 

and 
$$
\frac{d\vec{v}}{dt} = 0
$$

In [45]:
import boris_pusher
e_charge = 1.602176634e-19 # C, charge on an electron in Coulombs
amu = 1.66053906660e-27 # kg, atomic mass unit
q_ion = plasma_vars.q_ion * e_charge # C, charge per physical ion
m_ion = plasma_vars.m_ion * amu # kg
# Check that the pusher actually works
boris_pusher.test_gyration(q=q_ion, m=m_ion, B0=1.0, v0=2e4)

dt = boris_pusher.choose_dt(B, mesh, v_max=plasma_vars.v_ion_cathode, q=q_ion, m=m_ion)
boris_pusher.run_simulation(mesh, ptcls, B, cell_tags, physicalgroups, n_steps=2000, dt=dt, q=q_ion, m=m_ion)
# TODO Plot where the particles escape from (dist in collimator dt --> distribution on end wall?)
# Add constant particle stream per time step
# And how much thrust does that correlate to?

|v| drift: 7.28e-16 (should be small, eg ~1e-15). Type of v is <class 'numpy.ndarray'>
Larmor radius analytic 0.0064158 m, simulated 0.0064285 +/- 8.94e-06
B_max = 1.77 T | T_cyc = 1.14e-06 s | r_L(v_max) = 0.00271 m
h_min = 0.000677 m | dt_gyro = 5.68e-08 s | dt_cell = 2.26e-08 s
step      0 t=2.257e-08s  alive=801/1008  escaped=0 absorbed=207  <|v|>=15000  mean v=[14197.25408159   486.64023755    78.51416613]
step 19: no particles left


{'escaped': 0, 'absorbed': 1008}